The point of this notebook is to:
- determine the length of time it will take to use libraries to process text detection.
- test optimisation strategies.

Aims/requirements:
- All profanity should be filtered out, particularly those in subtitles. Starred words should also be removed.
- Profanity filtering should only take ~1 second per second of footage. This ensures an hour video can be filtered in ~1 hour
- Ideally, this process should take < 0.5 seconds per second of footage.

In [1]:
## Imports and global constants
import cv2
import easyocr, pytesseract, paddleocr
import time

video_path = "../data/raw/example1.mp4"

/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Load the profanity file

In [2]:
# Load profanity
profanity_set = set()
profanity_file_path = "../src/content_filter/config/profanity_words.txt"
with open(profanity_file_path, "r", encoding="utf-8") as f:
    for line in f:
        word = line.strip().lower()
        profanity_set.add(word)

def contains_profanity(text):
    words = text.lower().split()
    return any(word.strip(".,!?") in profanity_set for word in words)

Define a function for benchmarking EasyOCR

In [ ]:
def run_easy_ocr_on_image(image_data):
    """
    Run EasyOCR on one image and return detected text and time taken.
    """
    if not hasattr(run_easy_ocr_on_image, "_reader"):
        run_easy_ocr_on_image._reader = easyocr.Reader(['en'], quantize=True)

    t0 = time.time()
    results = run_easy_ocr_on_image._reader.readtext(image_data)
    dt = time.time() - t0

    texts = [text for (_, text, _) in results]
    return texts, dt


def run_ocr_test(duration, ocr_fn):
    """
    Run an OCR function over frames from the video for `duration` seconds.
    ocr_fn should accept image data and return (texts, dt).
    """
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS: {fps}")

    frame_count = 0
    max_frames = int(fps * duration)

    all_texts = []
    total_ocr_time = 0.0

    t0 = time.time()
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        texts, dt = ocr_fn(frame)
        all_texts.append(texts)
        total_ocr_time += dt

        frame_count += 1

    total_wall_time = time.time() - t0
    cap.release()
    cv2.destroyAllWindows()

    return all_texts, total_wall_time, total_ocr_time

def display_ocr_test_results(duration, all_texts, total_ocr_time, total_wall_time=None):
    print(f"It took: {total_wall_time:.2f} seconds to process all of the footage")
    print(f"Pure OCR time per second of footage: {(total_ocr_time/duration):.2f} seconds")
    if total_wall_time is not None:
        print(f"Wall time per second of footage: {total_wall_time/duration:.2f}s")
    print(f"Detected text items: {sum(map(lambda ts: len(ts), all_texts))}")
    print("Detected texts:")
    if len(all_texts) == 1:
        for text in all_texts[0]:
            print(f"- {text}")
    else:
        for i in range(len(all_texts)):
            print(f"-  Frame {i}: {all_texts[i]}")

In [ ]:
duration = 0.3
texts, wall_dt, ocr_dt = run_ocr_test(duration, run_easy_ocr_on_image)

FPS: 30.0


/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
display_ocr_test_results(duration, texts, ocr_dt, wall_dt)

It took: 59.64 seconds to process all of the footage
Pure OCR time per second of footage: 198.61 seconds
Wall time per second of footage: 198.81s
Detected text items: 157
Detected texts:
  Frame 0: bad*YHBC
  Frame 1: AAAAAAGH
  Frame 2: badioyaaW
  Frame 3: AAAAAAGH
  Frame 4: bad624FBD
  Frame 5: AAAAAAGH
  Frame 6: Da42047a0
  Frame 7: AAAAAAGH
  Frame 8: DaC1DO4NZC
  Frame 9: AAAAAAGH
  Frame 10: DaC1DO4NZC
  Frame 11: AAAAAAGH
  Frame 12: DaCDOYINAC
  Frame 13: AAAAAAGH
  Frame 14: DaUDUYInE0
  Frame 15: AAAAAAGH
  Frame 16: AULUEINHLU
  Frame 17: AAAAAAGH


#### Findings for naive method using EasyOCR on CPU:
- Time taken per second of footage: 190-200s
- <3 minutes per second of footage is WAY too much time.
- Cannot use GPU as the project should run locally on a machine.

In [ ]:
### Trying Tesseract ###
def run_tesseract_ocr(seconds, output_path):
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    profanity_boxes = []
    all_texts = []

    fps = cap.get(cv2.CAP_PROP_FPS)
    max_frames = int(fps * seconds)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter.fourcc('m', 'p', '4', 'v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    t0 = time.time()
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        texts, _ = run_tesseract_ocr(frame)
        all_texts.extend(texts)

        # Get detailed OCR data for profanity boxing
        data = pytesseract.image_to_data(frame, output_type=pytesseract.Output.DICT)
        n_boxes = len(data["text"])

        for i in range(n_boxes):
            word = data["text"][i]
            conf = int(data["conf"][i])

            if conf > 60 and contains_profanity(word):
                x = data["left"][i]
                y = data["top"][i]
                w = data["width"][i]
                h = data["height"][i]

                profanity_boxes.append((frame_count, (x, y, w, h)))
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

        frame_count += 1
        out.write(frame)

    dt = time.time() - t0
    cap.release()
    out.release()
    cv2.destroyAllWindows()

    return all_texts, dt


In [6]:
# dt, _ = run_tesseract_ocr(3, "../data/processed/censored_text.mp4")

In [7]:
# print(f"It took: {dt}")
# print(f"Time taken per second of footage: {(dt)/3 :.2f}s")

#### Tesseract naive findings
- Time taken per second of footage: 8.85s on average
- Still not fast enough: a 1 hour video will take 9 hours to filter, which is still too long.
- Need to judge accuracy

##### Testing quality of approaches: naive
In theory, the naive approach should be the most accurate as we are running the OCR every frame.

In [ ]:
def run_tesseract_ocr_on_image(image_data):
    """
    Use Tesseract OCR to extract all text and bounding boxes from an image.
    """
    # Convert to grayscale for better OCR accuracy
    if len(image_data.shape) == 3:
        gray = cv2.cvtColor(image_data, cv2.COLOR_BGR2GRAY)
    else:
        gray = image_data

    t0 = time.time()
    data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)
    dt = time.time() - t0

    texts = []
    num_boxes = len(data["text"])
    for i in range(num_boxes):
        text = data["text"][i].strip()
        if text:
            texts.append(text)

    return texts, dt

# image_path = "../data/samples/Image1.png" 

# image = cv2.imread(image_path)
# assert image is not None, f"Could not load image from: {image_path}"

# texts, dt = get_text_on_image(image)

# print(f"Found {len(texts)} text region(s) in {dt:.4f}s:\n")
# for t in texts:
#     print(f"  {t!r}")

There were many more text regions that were missed, including a subtitle region, which contained large text. Tesseract OCR on its own is not accurate enough for the project. TesseractOCR was built for text recognition in documents, not for text with messy backgrounds. The same text was detected when detecting by character instead, so no performance improvement there.<br>
We need to use a more powerful model but on a smaller area of the image. 

The main models for scene text detection are PaddleOCR (optimised for CPU) and EasyOCR (optimised for GPU).<br>
PaddleOCR is estimated to give a ~3-4x speed up on CPU, so I will test its speed and performance. Also worth testing EasyOCR performance.

In [ ]:
def run_paddle_ocr_test(image_path):
    """
    Run PaddleOCR on the given image data and return detected text and time taken.
    """
    paddle_ocr = paddleocr.PaddleOCR(lang='en')
    
    t0 = time.time()
    results = paddle_ocr.predict(image_path)
    dt = time.time() - t0

    texts = []
    if results and results[0]:
        for line in results[0]:
            text, confidence = line[1]
            texts.append(text)

    return texts, dt

# image_path = "../data/samples/Image1.png"
# image = cv2.imread(image_path)
# assert image is not None, f"Could not load image from: {image_path}"

# texts, dt = run_paddle_ocr_test(image)
# print(f"Time taken: {dt:.4f}s")
# print(f"Detected {len(texts)} text region(s):")
# for t in texts:
#     print(f"  {t!r}")


/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_serv

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)


Unfortunately, PaddleOCR is having many issues despite for re-installing it. I will use EasyOCR for now with heavy optimisations due to time I have left to spend on this.

In [15]:
image_path = "../data/samples/Image3.png"
image = cv2.imread(image_path)
assert image is not None, f"Could not load image from: {image_path}"

texts, dt = run_easy_ocr_on_image(image)
print(f"Time taken: {dt:.4f}s")
print(f"Detected {len(texts)} text region(s):")
for t in texts:
    print(f"- {t!r}")


Time taken: 7.4410s
Detected 7 text region(s):
- 'subgoal17,005/18,000'
- 'btch'
- 'BadboyHalo'
- 'substoday638'
- 'whatever you'
- 'is the word_'
- 'put'


##### Image 1
Detected 9 text region(s):
- 'Lange Chest'
- 'subgoal:16,994/18,000'
- 'Inventony'
- '29_'
- '12'
- 'substoday 610'
- 'veah i saw vou had'
- '00.00.22'
- '00.03.43'

##### Image 2
Detected 5 text region(s):
- 'subgoal; 16,994F'
- '18,955'
- 'subs'
- 'do-you wanna know the nickname'
- '00.00.39'

EasyOCR is good at detecting and generally recognising all the text. There are some errors, such as "Inventony" as opposed to "Inventory", and "Lange" instead of "Large". The mistakes come from when text is blurrier or smaller. The rest of the text detected is generally correct and by-in-large nothing was missed. It reads subtitles pretty much perfectly, which is the most important text to read.

## Improvements to consider
The main time optimisations are:
- only running EasyOCR on a pre-defined "subtitles region" and only running it when the transcription detects swearing. This will reduce the number of calls to EasyOCR significantly.
- Detecting text on the remaining screen using a more lightweight text detector every couple of frames. EasyOCR will then be called if text is detected.
- Reducing image resolution.

The main performance optimisations are:
- Increasing contrast to make text predictions more accurate/reliable

The other things I need to consider:
- Profanity filtering not as simple as checking if detected text is in profanity list. Need to account for starred words, slight inaccuracies in detection, etc.

In [1]:
def run_easy_ocr_on_image_optimised(image_data):
    """
    Run EasyOCR on one image and return detected text and time taken.
    Now we will include optimisations to improve accuracy and see how it affects performance.
    """
    if not hasattr(run_easy_ocr_on_image, "_reader"):
        run_easy_ocr_on_image._reader = easyocr.Reader(['en'], quantize=True)

    t0 = time.time()
    image_data = cv2.GaussianBlur(cv2.equalizeHist(image_data), (5, 5), 1)
    results = run_easy_ocr_on_image._reader.readtext(image_data)
    dt = time.time() - t0

    texts = [text for (_, text, _) in results]
    return texts, dt